<a href="https://colab.research.google.com/github/urvashi5555/Fake-news-detection-comparing-classical-ML-neural-networks-and-transformers/blob/main/03_Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import re
import string
import nltk
import tensorflow as tf

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.layers import (
    Input,
    Embedding,
    Dense,
    Dropout,
    LayerNormalization,
    MultiHeadAttention,
    GlobalAveragePooling1D
)

from tensorflow.keras.models import Model

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [ ]:
fake = pd.read_csv("Fake.csv")

real = pd.read_csv("True.csv")

fake["label"] = 0
real["label"] = 1

fake["content"] = (
    fake["title"] + " " + fake["text"]
)

real["content"] = (
    real["title"] + " " + real["text"]
)

news_df = pd.concat(
    [fake, real],
    ignore_index=True
)

news_df = news_df[
    ["content", "label"]
]

print(news_df.shape)

(44898, 2)


In [ ]:
covid_train = pd.read_csv(
    "Corona_NLP_train.csv",
    encoding='latin1'
)

covid_test = pd.read_csv(
    "Corona_NLP_test (1).csv",
    encoding='latin1'
)

covid_df = pd.concat(
    [covid_train, covid_test],
    ignore_index=True
)

print(covid_df.head())

print(covid_df.columns)

   UserName  ScreenName   Location     TweetAt  \
0      3799       48751     London  16-03-2020   
1      3800       48752         UK  16-03-2020   
2      3801       48753  Vagabonds  16-03-2020   
3      3802       48754        NaN  16-03-2020   
4      3803       48755        NaN  16-03-2020   

                                       OriginalTweet           Sentiment  
0  @MeNyrbie @Phil_Gahan @Chrisitv https://t.co/i...             Neutral  
1  advice Talk to your neighbours family to excha...            Positive  
2  Coronavirus Australia: Woolworths to give elde...            Positive  
3  My food stock is not the only one which is emp...            Positive  
4  Me, ready to go at supermarket during the #COV...  Extremely Negative  
Index(['UserName', 'ScreenName', 'Location', 'TweetAt', 'OriginalTweet',
       'Sentiment'],
      dtype='object')


In [ ]:
# REMOVE NEUTRAL
covid_df = covid_df[
    covid_df["Sentiment"] != "Neutral"
]

# LABEL MAPPING
label_map = {
    "Extremely Negative": 0,
    "Negative": 0,
    "Positive": 1,
    "Extremely Positive": 1
}

covid_df["label"] = (
    covid_df["Sentiment"]
    .map(label_map)
)

# KEEP ONLY TEXT + LABEL
covid_df = covid_df[
    ["OriginalTweet", "label"]
]

# RENAME COLUMN
covid_df.columns = [
    "content",
    "label"
]

print(covid_df.shape)

print(covid_df["label"].value_counts())

(36623, 2)
label
1    19592
0    17031
Name: count, dtype: int64


In [ ]:
combined_df = pd.concat(
    [news_df, covid_df],
    ignore_index=True
)

combined_df = combined_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print(combined_df.shape)

(81521, 2)


In [ ]:
combined_df = combined_df.sample(
    n=50000,
    random_state=42
)

combined_df = combined_df.reset_index(
    drop=True
)

print(combined_df.shape)

print(
    combined_df["label"].value_counts()
)

(50000, 2)
label
1    25123
0    24877
Name: count, dtype: int64


In [ ]:
combined_df = combined_df.sample(
    n=50000,
    random_state=42
)

combined_df = combined_df.reset_index(
    drop=True
)

print(combined_df.shape)

print(
    combined_df["label"].value_counts()
)

(50000, 2)
label
1    25123
0    24877
Name: count, dtype: int64


In [ ]:
stop_words = set(
    stopwords.words('english')
)

lemmatizer = WordNetLemmatizer()

def clean_text(text):

    text = str(text)

    text = text.lower()

    text = re.sub(
        r"http\S+|www\S+|https\S+",
        '',
        text
    )

    text = re.sub(
        r'@\w+',
        '',
        text
    )

    text = re.sub(
        r'#',
        '',
        text
    )

    text = re.sub(
        r'<.*?>',
        '',
        text
    )

    text = re.sub(
        r'\d+',
        '',
        text
    )

    text = text.translate(
        str.maketrans(
            '',
            '',
            string.punctuation
        )
    )

    text = re.sub(
        r'\s+',
        ' ',
        text
    ).strip()

    words = text.split()

    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words
    ]

    return " ".join(words)

combined_df["content"] = (
    combined_df["content"]
    .apply(clean_text)
)

print("Cleaning completed")

Cleaning completed


In [ ]:
X = combined_df["content"]

y = combined_df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", len(X_train))

print("Test size:", len(X_test))

Train size: 40000
Test size: 10000


In [ ]:
VOCAB_SIZE = 25000

tokenizer = Tokenizer(
    num_words=VOCAB_SIZE
)

tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(
    X_train
)

X_test_seq = tokenizer.texts_to_sequences(
    X_test
)

print("Tokenization completed")

Tokenization completed


In [ ]:
MAX_LEN = 150

X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=MAX_LEN
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=MAX_LEN
)

print(X_train_pad.shape)

print(X_test_pad.shape)

(40000, 150)
(10000, 150)


In [ ]:
EMBED_DIM = 64
NUM_HEADS = 4
FF_DIM = 128

# INPUT
inputs = Input(shape=(MAX_LEN,))

# EMBEDDING
embedding_layer = Embedding(
    input_dim=VOCAB_SIZE,
    output_dim=EMBED_DIM
)(inputs)

# MULTI-HEAD SELF ATTENTION
attention_output = MultiHeadAttention(
    num_heads=NUM_HEADS,
    key_dim=128
)(
    embedding_layer,
    embedding_layer
)

# DROPOUT
attention_output = Dropout(0.2)(
    attention_output
)

# RESIDUAL + LAYER NORMALIZATION
x = LayerNormalization(
    epsilon=1e-6
)(
    embedding_layer + attention_output
)

# FEED FORWARD NETWORK
ffn = Dense(
    128,
    activation='relu'
)(x)

ffn = Dense(64)(ffn)

# DROPOUT
ffn = Dropout(0.2)(ffn)

# SECOND RESIDUAL + NORMALIZATION
x = LayerNormalization(
    epsilon=1e-6
)(
    x + ffn
)

# GLOBAL POOLING
x = GlobalAveragePooling1D()(x)

# CLASSIFICATION HEAD
x = Dense(
    64,
    activation='relu'
)(x)

x = Dropout(0.2)(x)

# OUTPUT
outputs = Dense(
    1,
    activation='sigmoid'
)(x)

# BUILD MODEL
transformer_model = Model(
    inputs=inputs,
    outputs=outputs
)

# COMPILE
transformer_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

transformer_model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 150)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, 150, 64)   │  1,600,000 │ input_layer_2[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 150, 64)   │    132,672 │ embedding_2[0][0… │
│ (MultiHeadAttentio… │                   │            │ embedding_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_7 (Dropout) │ (None, 150, 64)   │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_4 (Add)         │ (None, 150, 64)   │          0 │ embedding_2[0][0… │
│                     │                   │            │ dropout_7[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 150, 64)   │        128 │ add_4[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 150, 128)  │      8,320 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 150, 64)   │      8,256 │ dense_7[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_8 (Dropout) │ (None, 150, 64)   │          0 │ dense_8[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_5 (Add)         │ (None, 150, 64)   │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_8[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 150, 64)   │        128 │ add_5[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 64)        │          0 │ layer_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_9 (Dense)     │ (None, 64)        │      4,160 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_9 (Dropout) │ (None, 64)        │          0 │ dense_9[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 1)         │         65 │ dropout_9[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,753,729 (6.69 MB)

 Trainable params: 1,753,729 (6.69 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history_transformer = transformer_model.fit(
    X_train_pad,
    y_train,
    epochs=3,
    batch_size=64,
    validation_split=0.1
)

Epoch 1/3
563/563 ━━━━━━━━━━━━━━━━━━━━ 21s 24ms/step - accuracy: 0.8314 - loss: 0.3245 - val_accuracy: 0.8932 - val_loss: 0.2433
Epoch 2/3
563/563 ━━━━━━━━━━━━━━━━━━━━ 8s 13ms/step - accuracy: 0.9303 - loss: 0.1722 - val_accuracy: 0.9118 - val_loss: 0.2175
Epoch 3/3
563/563 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step - accuracy: 0.9536 - loss: 0.1210 - val_accuracy: 0.9010 - val_loss: 0.2865


In [ ]:
transformer_pred = transformer_model.predict(
    X_test_pad
)

transformer_pred = (
    transformer_pred > 0.5
).astype(int)

print("Predictions completed")

313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step
Predictions completed


In [ ]:
transformer_accuracy = accuracy_score(
    y_test,
    transformer_pred
)

print(
    "Transformer Test Accuracy:",
    transformer_accuracy
)

print(
    "Transformer Test Accuracy (%):",
    transformer_accuracy * 100
)

Transformer Test Accuracy: 0.9072
Transformer Test Accuracy (%): 90.72


In [ ]:
history_transformer = transformer_model.fit(
    X_train_pad,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.1
)

Epoch 1/5
563/563 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step - accuracy: 0.9656 - loss: 0.0921 - val_accuracy: 0.9057 - val_loss: 0.2918
Epoch 2/5
563/563 ━━━━━━━━━━━━━━━━━━━━ 8s 13ms/step - accuracy: 0.9739 - loss: 0.0701 - val_accuracy: 0.9057 - val_loss: 0.3644
Epoch 3/5
563/563 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step - accuracy: 0.9792 - loss: 0.0571 - val_accuracy: 0.9075 - val_loss: 0.4154
Epoch 4/5
563/563 ━━━━━━━━━━━━━━━━━━━━ 8s 13ms/step - accuracy: 0.9823 - loss: 0.0466 - val_accuracy: 0.9070 - val_loss: 0.4481
Epoch 5/5
563/563 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step - accuracy: 0.9858 - loss: 0.0369 - val_accuracy: 0.9060 - val_loss: 0.5327


In [ ]:
transformer_pred = transformer_model.predict(
    X_test_pad
)

transformer_pred = (
    transformer_pred > 0.5
).astype(int)

transformer_accuracy = accuracy_score(
    y_test,
    transformer_pred
)

print(
    "Transformer Test Accuracy:",
    transformer_accuracy
)

print(
    "Transformer Test Accuracy (%):",
    transformer_accuracy * 100
)

313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step
Transformer Test Accuracy: 0.9042
Transformer Test Accuracy (%): 90.42


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

In [ ]:
transformer_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.0003
    ),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history_transformer = transformer_model.fit(
    X_train_pad,
    y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop]
)

Epoch 1/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step - accuracy: 0.9936 - loss: 0.0130 - val_accuracy: 0.9062 - val_loss: 0.7816
Epoch 2/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step - accuracy: 0.9954 - loss: 0.0094 - val_accuracy: 0.9070 - val_loss: 0.9060


In [ ]:
transformer_pred = transformer_model.predict(
    X_test_pad
)

transformer_pred = (
    transformer_pred > 0.5
).astype(int)

print("Predictions completed")

313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step
Predictions completed


In [ ]:
transformer_accuracy = accuracy_score(
    y_test,
    transformer_pred
)

print(
    "Transformer Test Accuracy:",
    transformer_accuracy
)

print(
    "Transformer Test Accuracy (%):",
    transformer_accuracy * 100
)

Transformer Test Accuracy: 0.9042
Transformer Test Accuracy (%): 90.42


In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        transformer_pred
    )
)

              precision    recall  f1-score   support

           0       0.91      0.90      0.90      4975
           1       0.90      0.91      0.91      5025

    accuracy                           0.90     10000
   macro avg       0.90      0.90      0.90     10000
weighted avg       0.90      0.90      0.90     10000



In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_test,
    transformer_pred
)

print(cm)

[[4467  508]
 [ 450 4575]]
